In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import gamma
import math
import time

# Dados de treino originais (Naturais de 1 a 10)
X_train = np.arange(1, 11, dtype=float)
y_factorial = np.array([math.factorial(int(n)) for n in X_train], dtype=float)

# Transformação de escala de Chang/Wiktorowicz para o espaço intermediário
y_train_transformed = np.zeros_like(X_train)
for i, n in enumerate(X_train):
    fact = math.factorial(int(n))
    y_train_transformed[i] = 1.0 if fact == 1 else 1.0 / np.log(fact)

print("X de Treino:", X_train)
print("Y Transformado:", y_train_transformed)

In [ ]:
def tsk_inference(X, centers, sigmas, y_target=None, ridge_lambda=1.0):
    X = np.atleast_1d(X)
    N = len(X)
    R = len(centers) # Será setado para 12 conforme exigência da lauda fuzzy
    
    # 1. Fuzzificação (Antecedentes Gaussianos)
    W = np.zeros((N, R))
    for i in range(N):
        for j in range(R):
            W[i, j] = np.exp(-((X[i] - centers[j])**2) / (2 * (sigmas[j]**2) + 1e-5))
            
    row_sums = W.sum(axis=1, keepdims=True)
    W_norm = np.where(row_sums > 1e-12, W / row_sums, 1.0 / R)
    
    # 2. Matriz de Design Global para os Consequentes Lineares
    X_hat = np.zeros((N, 2 * R))
    for j in range(R):
        X_hat[:, 2*j] = W_norm[:, j] * X
        X_hat[:, 2*j + 1] = W_norm[:, j]
        
    # 3. Solução Analítica Estável via Ridge (Wiktorowicz, 2021)
    if y_target is not None:
        A = X_hat.T @ X_hat + ridge_lambda * np.eye(2 * R)
        try:
            P = np.linalg.inv(A) @ X_hat.T @ y_target
        except np.linalg.LinAlgError:
            P = np.linalg.pinv(A) @ X_hat.T @ y_target
        return X_hat @ P, P
    else:
        return X_hat

In [ ]:
# Universo de Discurso Expandido Autoral
X_MIN_EXPANDIDO = -10.0
X_MAX_EXPANDIDO = 10.0

def particle_swarm_optimization(X, y_trans, num_rules=12, pop_size=40, iterations=40, seed=42):
    np.random.seed(seed)
    # Centros mapeando o domínio estendido completo para as 12 regras legítimas
    centers = np.linspace(X_MIN_EXPANDIDO, X_MAX_EXPANDIDO, num_rules)
    
    w, c1, c2 = 0.6, 1.8, 1.8
    position = np.random.uniform(1.5, 5.0, size=(pop_size, num_rules))
    velocity = np.random.uniform(-0.2, 0.2, size=(pop_size, num_rules))
    
    pbest_position = position.copy()
    pbest_fitness = np.array([np.mean((y_trans - tsk_inference(X, centers, ind, y_trans)[0])**2) for ind in position])
    gbest_idx = np.argmin(pbest_fitness)
    gbest_position = pbest_position[gbest_idx].copy()
    
    history = []
    for it in range(iterations):
        for i in range(pop_size):
            r1, r2 = np.random.rand(num_rules), np.random.rand(num_rules)
            velocity[i] = w * velocity[i] + c1 * r1 * (pbest_position[i] - position[i]) + c2 * r2 * (gbest_position - position[i])
            position[i] = np.clip(position[i] + velocity[i], 1.2, 7.0) # Limite inferior estrito de estabilidade
            
            y_pred, _ = tsk_inference(X, centers, position[i], y_trans)
            current_mse = np.mean((y_trans - y_pred)**2)
            
            if current_mse < pbest_fitness[i]:
                pbest_fitness[i] = current_mse
                pbest_position[i] = position[i].copy()
                if current_mse < pbest_fitness[gbest_idx]:
                    gbest_position = position[i].copy()
                    gbest_idx = i
        history.append(pbest_fitness[gbest_idx])
    return gbest_position, history, centers

def genetic_algorithm(X, y_trans, num_rules=12, pop_size=40, generations=40, mutation_rate=0.2, seed=42):
    np.random.seed(seed)
    centers = np.linspace(X_MIN_EXPANDIDO, X_MAX_EXPANDIDO, num_rules)
    population = np.random.uniform(1.5, 5.0, size=(pop_size, num_rules))
    history = []
    
    for gen in range(generations):
        fitness = np.array([np.mean((y_trans - tsk_inference(X, centers, ind, y_trans)[0])**2) for ind in population])
        best_idx = np.argmin(fitness)
        history.append(fitness[best_idx])
        
        new_population = []
        for _ in range(pop_size):
            candidates = np.random.choice(pop_size, size=3, replace=False)
            winner = candidates[np.argmin(fitness[candidates])]
            new_population.append(population[winner].copy())
            
        new_population = np.array(new_population)
        for i in range(0, pop_size, 2):
            if i+1 < pop_size and np.random.rand() < 0.8:
                alpha = np.random.rand()
                child1 = alpha * new_population[i] + (1 - alpha) * new_population[i+1]
                child2 = alpha * new_population[i+1] + (1 - alpha) * new_population[i]
                new_population[i] = np.clip(child1, 1.2, 7.0)
                new_population[i+1] = np.clip(child2, 1.2, 7.0)
                
        for i in range(pop_size):
            if np.random.rand() < mutation_rate:
                mutation_vector = np.random.normal(0, 0.2, size=num_rules)
                new_population[i] = np.clip(new_population[i] + mutation_vector, 1.2, 7.0)
                
        new_population[0] = population[best_idx]
        population = new_population
        
    final_fitness = np.array([np.mean((y_trans - tsk_inference(X, centers, ind, y_trans)[0])**2) for ind in population])
    return population[np.argmin(final_fitness)], history, centers

In [ ]:
seeds = [10, 42, 100, 2026, 999]
NUM_REGRAS = 12 # 12 Regras obrigatórias validadas!
ITERACOES = 40

pso_mses, pso_rmses, pso_mapes, pso_times, pso_curves = [], [], [], [], []
ga_mses, ga_rmses, ga_mapes, ga_times, ga_curves = [], [], [], [], []

def calculate_advanced_metrics(y_real, y_pred_transformed):
    # Inversão exponencial estável
    y_pred_real = np.exp(1.0 / y_pred_transformed)
    mse = np.mean((y_real - y_pred_real)**2)
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((y_real - y_pred_real) / y_real)) * 100
    return mse, rmse, mape

# --- EXECUÇÃO PSO ---
for s in seeds:
    t0 = time.time()
    best_sigmas, hist, centers = particle_swarm_optimization(X_train, y_train_transformed, num_rules=NUM_REGRAS, iterations=ITERACOES, seed=s)
    pso_times.append(time.time() - t0)
    pso_curves.append(hist)
    
    y_pred_trans, _ = tsk_inference(X_train, centers, best_sigmas, y_train_transformed)
    _, rmse, mape = calculate_advanced_metrics(y_factorial, y_pred_trans)
    pso_mses.append(hist[-1])
    pso_rmses.append(rmse)
    pso_mapes.append(mape)

# --- EXECUÇÃO GA ---
for s in seeds:
    t0 = time.time()
    best_sigmas, hist, centers = genetic_algorithm(X_train, y_train_transformed, num_rules=NUM_REGRAS, generations=ITERACOES, seed=s)
    ga_times.append(time.time() - t0)
    ga_curves.append(hist)
    
    y_pred_trans, _ = tsk_inference(X_train, centers, best_sigmas, y_train_transformed)
    _, rmse, mape = calculate_advanced_metrics(y_factorial, y_pred_trans)
    ga_mses.append(hist[-1])
    ga_rmses.append(rmse)
    ga_mapes.append(mape)

print("\n" + "="*60)
print("     TABELA COMPARATIVA ESTATÍSTICA COM NOVAS MÉTRICAS")
print("="*60)
print(f"Métrica               | PSO              | GA")
print(f"------------------------------------------------------------")
print(f"Melhor MSE (Treino)   | {np.min(pso_mses):.6f}         | {np.min(ga_mses):.6f}")
print(f"Desvio Padrão MSE     | {np.std(pso_mses):.6f}         | {np.std(ga_mses):.6f}")
print(f"Média RMSE (Real)     | {np.mean(pso_rmses):.2f}       | {np.mean(ga_rmses):.2f}")
print(f"Média MAPE (Real %)   | {np.mean(pso_mapes):.2f}%          | {np.mean(ga_mapes):.2f}%")
print(f"Tempo Médio (s)       | {np.mean(pso_times):.4f}           | {np.mean(ga_times):.4f}")
print("="*60)

In [ ]:
# =====================================================================
# CÉLULA 5: PLOTAGEM SEPARADA POR DOMÍNIOS (POSITIVOS VS NEGATIVOS)
# =====================================================================

# 1. Recuperação dos melhores modelos de cada categoria (PSO e GA)
best_pso_idx = np.argmin(pso_mses)
best_pso_sigmas, _, centers_pso = particle_swarm_optimization(X_train, y_train_transformed, num_rules=NUM_REGRAS, iterations=ITERACOES, seed=seeds[best_pso_idx])
_, optimal_P_pso = tsk_inference(X_train, centers_pso, best_pso_sigmas, y_train_transformed)

best_ga_idx = np.argmin(ga_mses)
best_ga_sigmas, _, ga_centers = genetic_algorithm(X_train, y_train_transformed, num_rules=NUM_REGRAS, generations=ITERACOES, seed=seeds[best_ga_idx])
_, optimal_P_ga = tsk_inference(X_train, ga_centers, best_ga_sigmas, y_train_transformed)

# 2. Geração dos domínios contínuos separados para análise visual precisa
X_pos_continuous = np.linspace(1.0, 10.0, 200)
X_neg_continuous = np.linspace(-10.0, 0.9, 300) # Vai do limite inferior até próximo a 1

# Inferências para o Domínio Positivo
y_pso_pos_trans = tsk_inference(X_pos_continuous, centers_pso, best_pso_sigmas) @ optimal_P_pso
y_ga_pos_trans = tsk_inference(X_pos_continuous, ga_centers, best_ga_sigmas) @ optimal_P_ga
pso_factorial_pos = np.exp(1.0 / y_pso_pos_trans)
ga_factorial_pos = np.exp(1.0 / y_ga_pos_trans)

# Inferências para o Domínio Negativo (A Extrapolação Inédita)
y_pso_neg_trans = tsk_inference(X_neg_continuous, centers_pso, best_pso_sigmas) @ optimal_P_pso
y_ga_neg_trans = tsk_inference(X_neg_continuous, ga_centers, best_ga_sigmas) @ optimal_P_ga
pso_factorial_neg = np.exp(1.0 / y_pso_neg_trans)
ga_factorial_neg = np.exp(1.0 / y_ga_neg_trans)

# 3. Configuração da Grade de Gráficos (2x2)
fig, axs = plt.subplots(2, 2, figsize=(16, 11))

# --- QUADRANTE (0, 0): VELOCIDADE DE CONVERGÊNCIA ---
axs[0, 0].plot(np.mean(pso_curves, axis=0), 'r-', linewidth=2, label='PSO (Média)')
axs[0, 0].plot(np.mean(ga_curves, axis=0), 'b-', linewidth=2, label='GA (Média)')
axs[0, 0].set_title("Velocidade de Convergência: PSO vs GA")
axs[0, 0].set_xlabel("Iterações / Gerações")
axs[0, 0].set_ylabel("MSE Médio (Espaço Transformado)")
axs[0, 0].set_yscale('log')
axs[0, 0].grid(True)
axs[0, 0].legend()

# --- QUADRANTE (0, 1): ESPAÇO INTERMEDIÁRIO DE REGRESSÃO ---
y_gamma_trans_pos = 1.0 / np.log(gamma(X_pos_continuous + 1))
axs[0, 1].plot(X_pos_continuous, y_gamma_trans_pos, 'g-', label='Meta Analítica $1/\\ln(x!)$', alpha=0.5)
axs[0, 1].plot(X_pos_continuous, y_pso_pos_trans, 'r--', label='Inferência PSO')
axs[0, 1].plot(X_pos_continuous, y_ga_pos_trans, 'b:', label='Inferência GA', linewidth=2)
axs[0, 1].scatter(X_train, y_train_transformed, color='black', zorder=5, label='Treino')
axs[0, 1].set_title("Superfície de Inferência no Espaço Transformado")
axs[0, 1].set_xlabel("Entrada ($x$)")
axs[0, 1].set_ylabel("Valor Transformado")
axs[0, 1].grid(True)
axs[0, 1].legend()

# --- QUADRANTE (1, 0): COMPORTAMENTO NOS NÚMEROS POSITIVOS ---
axs[1, 0].plot(X_pos_continuous, gamma(X_pos_continuous + 1), 'g-', label='Função Gamma $\\Gamma(x+1)$', alpha=0.6)
axs[1, 0].plot(X_pos_continuous, pso_factorial_pos, 'r--', label='Reconstrução PSO')
axs[1, 0].plot(X_pos_continuous, ga_factorial_pos, 'b:', label='Reconstrução GA', linewidth=2)
axs[1, 0].scatter(X_train, y_factorial, color='black', zorder=5, label='Fatoriais Reais ($n!$)')
axs[1, 0].set_title("Aproximação Isolada: Domínio Positivo")
axs[1, 0].set_xlabel("Entrada ($x$)")
axs[1, 0].set_ylabel("Valor do Fatorial ($n!$) - Escala Log")
axs[1, 0].set_yscale('log')
axs[1, 0].grid(True)
axs[1, 0].legend()

# --- QUADRANTE (1, 1): COMPORTAMENTO EXTRAPOLADO NOS NEGATIVOS ---
axs[1, 1].plot(X_neg_continuous, pso_factorial_neg, 'r--', linewidth=2, label='Extrapolação PSO')
axs[1, 1].plot(X_neg_continuous, ga_factorial_neg, 'b:', linewidth=2, label='Extrapolação GA')
axs[1, 1].set_title("Predição Inédita Isolada: Domínio Negativo / Fracionário")
axs[1, 1].set_xlabel("Universo de Discurso Expandido ($x$)")
axs[1, 1].set_ylabel("Saída Reconstruída (Escala Linear)")
axs[1, 1].grid(True)
axs[1, 1].legend()

plt.tight_layout()
plt.show()